In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

In [ ]:
io_path = Path('..','data','in')

In [ ]:
def portfolio_variance(weights, returns, trading_days=252):
    return trading_days * np.dot(weights.T, np.dot(returns.cov(), weights))

def relative_risk_contributions(weights, returns):
    volatility = np.sqrt(portfolio_variance(weights, returns))
    covariance = returns.cov()
    marginal_vols = np.dot(covariance, weights) / volatility
    risk_contrib = marginal_vols * weights
    relative_risk_contrib = risk_contrib / risk_contrib.sum()
    return relative_risk_contrib

In [ ]:
df = pd.read_pickle(io_path / Path('sample_df.pickle'))

df_anagrafica = pd.read_excel(
    io_path / Path('demo.xlsx'),
    sheet_name=1,
    dtype=str
).rename(
    columns={
        'Exchange': 'exchange',
        'Ticker': 'ticker',
        'Security Name': 'name',
        'Asset Class': 'asset_class',
        'Macro Asset Class': 'macro_asset_class',
    }
)

In [ ]:
df_log_rets = np.log(df.div(df.shift()))

In [ ]:
df_log_rets.sample()

In [ ]:
weights = np.array(
    [
        52.4,
        1.7,
        0.6,
        16.3,
        0.3,
        2.0,
        5.7,
        8.3,
        3.3,
        5.5,
        2.4,
        1.6,
    ]
)

In [ ]:
rrc = relative_risk_contributions(
    weights=weights,
    returns=df_log_rets
)

In [ ]:
data = {
  'ticker': df.columns.to_list(),
  'relative_risk_contribution': rrc,
}

df_rrc = pd.DataFrame(data)

In [ ]:
df_anagrafica['ticker_'] = df_anagrafica['ticker'] + '.' + df_anagrafica['exchange']

In [ ]:
df_ = df_rrc.merge(
    df_anagrafica[['ticker_','asset_class','macro_asset_class']],
    left_on='ticker',
    right_on='ticker_',
).groupby(
    'asset_class', as_index=False,
)['relative_risk_contribution'].sum().sort_values('relative_risk_contribution', ascending=False)

In [ ]:
df_['relative_risk_contribution'] = 100 * df_['relative_risk_contribution']
df_['cumsum'] = df_['relative_risk_contribution'].cumsum()

df_['relative_risk_contribution'] = df_['relative_risk_contribution'].round(1)
df_['cumsum'] = df_['cumsum'].round(1)

In [ ]:
df_